In [1]:
import asf_search as asf        # search the ASF catalog
import earthaccess               # authenticate with NASA Earthdata
import h5py                      # read HDF5 files
import numpy as np               # array math
import rioxarray                 # geospatial arrays + GeoTIFF export
import xarray as xr              # labeled arrays
import matplotlib.pyplot as plt  # plotting
from datetime import datetime
import geopandas as gpd
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

print("✓ All imports OK")
print(f"  asf_search version: {asf.__version__}")

/bsuhome/julialober/miniforge3/envs/coherence_v2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports OK
  asf_search version: 12.0.7


In [2]:
# ── Helper: parse interferometric pair dates from GUNW scene name ──
def parse_gunw_dates(scene_name):
    """
    GUNW filenames encode the reference and secondary acquisition dates.
    Example: ...20251221T135101...20260102T135101...
             ↑ reference (Dec 21)   ↑ secondary (Jan 2)
    """
    parts = scene_name.split('_')
    timestamps = [p for p in parts if len(p) == 15 and 'T' in p]
    if len(timestamps) >= 4:
        return timestamps[0][:8], timestamps[2][:8]
    return None, None

def temporal_baseline(scene_name):
    ref, sec = parse_gunw_dates(scene_name)
    if ref and sec:
        return abs((datetime.strptime(sec,'%Y%m%d') -
                    datetime.strptime(ref,'%Y%m%d')).days)
    return None

print("Helper functions defined.")

Helper functions defined.


In [3]:
# Search for NISAR GUNW data on Track 77
print("Searching ASF catalog for NISAR GUNW on Track 77...")

results = asf.search(
    dataset         = 'NISAR',
    processingLevel = 'GUNW',
    relativeOrbit   = 77,
    maxResults      = 500,
)

print(f"\nFound {len(results)} granules on Track 77")

# What dates are available?
dates = sorted(set(
    parse_gunw_dates(r.properties['sceneName'])[0]
    for r in results
    if parse_gunw_dates(r.properties['sceneName'])[0]
))

print(f"\nReference dates available ({len(dates)} unique):")
for d in dates:
    fmt = f"{d[:4]}-{d[4:6]}-{d[6:]}"
    print(f"  {fmt}")

Searching ASF catalog for NISAR GUNW on Track 77...


ERROR:asf_search:The asf-search module ecountered an error with CMR,and the following message was automatically reported to ASF:

"
message
"If you have any questions email uso@asf.alaska.edu
ERROR:asf_search:The asf-search module ecountered an error with CMR,and the following message was automatically reported to ASF:

"
message
"If you have any questions email uso@asf.alaska.edu
ERROR:asf_search:The asf-search module ecountered an error with CMR,and the following message was automatically reported to ASF:

"
message
"If you have any questions email uso@asf.alaska.edu
ERROR:asf_search:Results may be incomplete due to a search error. See ASF_LOGGER logging for more details.



Found 329 granules on Track 77

Reference dates available (20 unique):
  2025-11-03
  2025-11-15
  2025-11-27
  2025-12-09
  2025-12-21
  2026-01-14
  2026-02-07
  2026-02-19
  2026-03-15
  2026-03-27
  2026-04-08
  2026-04-20
  2026-05-02
  2026-05-14
  2026-05-26
  2026-06-07
  2026-06-19
  2026-07-01
  2026-07-13
  2026-07-25
